<a href="https://colab.research.google.com/github/wavival/fundamentos-llms/blob/main/GPT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install einops xformers np

In [8]:
import numpy as np
# Importa el módulo np (normalmente se refiere a NumPy)
# NumPy se usa para operaciones matemáticas con arrays y álgebra lineal eficiente

import torch
# Importa PyTorch, librería principal para deep learning
# Permite crear tensores, redes neuronales y entrenamiento con autograd

import torch.nn as nn
# Importa el submódulo nn de PyTorch
# Contiene clases para construir Neural Networks (layers, losses, modules)

import torch.nn.functional as F
# Importa funciones funcionales de PyTorch
# Incluye operaciones sin estado como activations, loss functions, etc.

from torch.nn.modules import ModuleList
# Importa ModuleList desde PyTorch
# Es una lista especial de módulos (layers) que PyTorch registra automáticamente para entrenamiento

from torch.nn.modules.normalization import LayerNorm
# Importa LayerNorm específicamente desde el módulo de normalización
# Layer Normalization estabiliza la distribución de activaciones en redes profundas

from torch import nn
# Reimporta nn desde torch directamente
# Es redundante porque ya fue importado arriba; normalmente no se hace
# nn sigue siendo el namespace de neural network layers en PyTorch

import copy
# Importa módulo estándar de Python para copiar objetos
# Permite deep copy (copias independientes de estructuras complejas como modelos)

import math
# Importa librería matemática estándar de Python
# Proporciona funciones como sqrt, sin, cos, log, etc. para cálculos numéricos

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Define el dispositivo (device) donde se ejecutarán los tensores y el modelo en PyTorch
# torch.device es un objeto que representa hardware de cómputo (CPU o GPU)

# "cuda" es el backend de NVIDIA GPU acceleration en PyTorch
# Permite ejecutar operaciones en GPU, mucho más rápido para deep learning

# torch.cuda.is_available() verifica si hay una GPU compatible con CUDA disponible en el sistema
# Devuelve True si existe GPU NVIDIA configurada correctamente, False si no

# "cpu" es el fallback (procesador central)
# Se usa si no hay GPU disponible o si CUDA no está instalado

# Expresión condicional:
# "cuda" if condition else "cpu"
# Si hay GPU → usa CUDA
# Si no hay GPU → usa CPU

In [10]:
device

# Resultado:
# device guarda algo como:
# device(type='cuda') o device(type='cpu')

# Este objeto se usa luego para mover modelos y tensores:
# model.to(device)
# tensor.to(device)

device(type='cuda')

In [11]:
def _get_clones(module, n):
    # Define una función auxiliar llamada _get_clones
    # Sirve para crear múltiples copias independientes de un mismo módulo (layer o bloque)

    return ModuleList([copy.deepcopy(module) for i in range(n)])
    # ModuleList: contenedor especial de PyTorch que almacena módulos (nn.Module)
    # y asegura que PyTorch los registre correctamente para entrenamiento

    # [copy.deepcopy(module) for i in range(n)]:
    # List comprehension en Python
    # Itera n veces (i va de 0 a n-1)
    # En cada iteración crea una copia profunda del módulo original

    # copy.deepcopy(module):
    # Crea una copia completamente independiente del módulo
    # No comparte memoria ni pesos con el original
    # Esto es clave para Transformers: cada capa debe aprender parámetros distintos

    # Resultado final:
    # Se devuelve una lista de n capas idénticas en arquitectura
    # pero con parámetros independientes para entrenamiento

In [12]:
class Conv1D(nn.Module):
    def __init__(self, nx, nf):
        # Define una clase Conv1D que hereda de nn.Module (base de todas las capas en PyTorch)
        # Esta capa implementa una proyección lineal tipo 1D convolution usada en Transformers

        '''
        nx: Numero de datos de entrada.
        nf: Numero de filtros. (Canales de salida).
        '''
        # nx: dimensionalidad de entrada (input feature size)
        # nf: número de filtros o features de salida (output channels)

        super().__init__()
        # Inicializa la clase base nn.Module
        # Necesario para registrar parámetros y funcionalidades de PyTorch

        self.nf = nf
        # Guarda el número de filtros de salida como atributo del objeto

        # Inicializando una matriz vacia de pesos del tamaño (nx)X(nf)
        w = torch.empty(nx, nf)
        # Crea un tensor vacío (sin valores inicializados) con forma [nx, nf]
        # Representa la matriz de pesos de la capa

        # Inicializando los pesos con una distribución normal.
        nn.init.normal_(w, std=0.02)
        # Inicializa los pesos con distribución normal (Gaussian)
        # std=0.02 controla la desviación estándar (pesos pequeños al inicio)
        # Esto ayuda a estabilidad en el entrenamiento

        # Calculando los pesos y sesgos encodeandos usando nn.Parameter
        self.weight = nn.Parameter(w)
        # Convierte el tensor w en un parámetro entrenable del modelo
        # PyTorch ahora lo actualizará durante backpropagation

        self.bias = nn.Parameter(torch.zeros(nf))
        # Inicializa el bias como vector de ceros de tamaño nf
        # También es un parámetro entrenable (se ajusta durante entrenamiento)

    def forward(self, x):
        # Define el forward pass (cómo fluye la información en la capa)
        # x es el tensor de entrada

        '''x:Tensor de entrada.'''
        # Documenta que x es un tensor con features en la última dimensión

        # El tamaño de la salida es la suma de la segunda dimensión de X y el número de filtros nf.
        size_out = x.size()[:-1] + (self.nf,)
        # x.size()[:-1] toma todas las dimensiones excepto la última (batch, seq, etc.)
        # Luego se agrega nf como nueva última dimensión de salida
        # Resultado: cambia el embedding size de nx → nf

        # Producto punto Q,K(Transpuesta) y V
        x = torch.addmm(
            self.bias,
            x.view(-1, x.size(-1)),
            self.weight
        )
        # torch.addmm realiza: bias + (matmul input @ weight)
        # Es equivalente a: xW + b en una capa lineal

        # x.view(-1, x.size(-1)) aplana las dimensiones anteriores
        # Convierte x en matriz 2D: (batch*seq, nx)

        # self.weight tiene forma (nx, nf)
        # resultado del matmul: (batch*seq, nf)

        # self.bias se suma automáticamente a cada fila (broadcasting)

        # x.view ayuda a calcular la transpuesta.
        # Aquí realmente se usa reshape implícito para restaurar estructura original

        x = x.view(*size_out)
        # Restaura la forma original del tensor con nueva dimensión nf
        # Ejemplo: (batch, seq, nx) → (batch, seq, nf)

        return x
        # Devuelve la salida final de la capa lineal proyectada

In [13]:
class FeedForward(nn.Module):
    # Define una clase FeedForward que hereda de nn.Module
    # Representa el MLP (Multi-Layer Perceptron) dentro de un Transformer
    # Es la parte que procesa cada token de forma independiente después de la atención

    def __init__(self, dropout, d_model=768, nx=768*4):
        # Constructor de la capa FeedForward

        super().__init__()
        # Inicializa la clase base nn.Module
        # Necesario para registrar parámetros correctamente en PyTorch

        self.c_fc = Conv1D(d_model, nx)
        # Primera proyección lineal (expansión de dimensión)
        # Convierte embeddings de tamaño d_model → nx (4x más grande típicamente)
        # Esto aumenta la capacidad de representación del modelo

        self.c_proj = Conv1D(nx, d_model)
        # Segunda proyección lineal (reducción de dimensión)
        # Convierte nx → d_model para volver al tamaño original del embedding

        self.act = F.gelu
        # Función de activación GELU (Gaussian Error Linear Unit)
        # Introduce no linealidad en el modelo
        # Es la activación estándar en Transformers tipo GPT

        self.dropout = nn.Dropout(dropout)
        # Dropout es una técnica de regularización
        # Apaga aleatoriamente neuronas durante entrenamiento
        # Ayuda a evitar overfitting

    def forward(self, x):
        # Define el forward pass del FeedForward
        # x es el tensor de entrada (batch, seq, d_model)

        return self.dropout(self.c_proj(self.act(self.c_fc(x))))
        # Flujo completo de datos:

        # 1. self.c_fc(x)
        #    Expande dimensión: d_model → nx
        #    Permite mayor capacidad de aprendizaje interno

        # 2. self.act(...)
        #    Aplica GELU elemento por elemento
        #    Introduce no linealidad (clave para redes profundas)

        # 3. self.c_proj(...)
        #    Reduce dimensión: nx → d_model
        #    Vuelve al espacio original del modelo

        # 4. self.dropout(...)
        #    Aplica regularización apagando neuronas aleatorias
        #    Mejora generalización del modelo

In [14]:
class Attention(nn.Module):
    def __init__(self, d_model=768, n_head=12, n_ctx=1024, d_head=64, bias=True, scale=False):
        # Define una clase Attention que hereda de nn.Module
        # Implementa Multi-Head Self-Attention estilo Transformer / GPT

        '''
        Params:
        d_model: Dimensión del embedding de entrada/salida del modelo (feature size global)
        n_head: Número de cabezas de atención (parallel attention mechanisms)
        n_ctx: Context length máximo (longitud de secuencia soportada)
        d_head: Dimensión por cada head (features por cabeza)
        bias: Flag para usar bias en capas lineales (no usado explícitamente aquí)
        scale: Controla si se aplica scaling por sqrt(dk) para estabilidad numérica
        '''

        super().__init__()
        # Inicializa clase base nn.Module
        # Necesario para registrar capas y buffers correctamente

        self.n_head = n_head
        # Guarda el número de heads de atención

        self.d_model = d_model
        # Guarda la dimensión total del embedding del modelo

        self.c_attn = Conv1D(d_model, d_model * 3)
        # Capa lineal que proyecta input a Q, K y V concatenados
        # Output size = 3 * d_model → luego se separa en Q, K, V

        self.scale = scale
        # Booleano que indica si se aplica scaling en atención

        self.softmax = nn.Softmax(dim=-1)
        # Softmax aplicado sobre la última dimensión (normaliza scores por fila)
        # Convierte logits en distribución de probabilidad

        self.register_buffer(
            "bias",
            torch.tril(torch.ones(n_ctx, n_ctx)).view(1, 1, n_ctx, n_ctx)
        )
        # Causal mask triangular inferior
        # torch.ones → matriz llena de 1
        # torch.tril → deja solo diagonal inferior (evita ver futuro)
        # view(1,1,n_ctx,n_ctx) → ajusta shape para broadcasting en atención

        self.dropout = nn.Dropout(0.1)
        # Dropout aplicado a los attention weights
        # Reduce overfitting apagando conexiones aleatorias

        self.c_proj = Conv1D(d_model, d_model)
        # Proyección final después de atención
        # Devuelve output al mismo tamaño que entrada (residual connection compatible)


    def split_heads(self, x):
        # Divide el tensor en múltiples heads de atención

        """
        Convierte el tensor en múltiples cabezas de atención.

        return shape:
        [batch, head, sequence, features_per_head]
        """

        new_shape = x.size()[:-1] + (self.n_head, x.size(-1) // self.n_head)
        # x.size()[:-1] → todas las dimensiones excepto última (batch, seq)
        # Se agrega:
        # (n_head, d_head)
        # donde d_head = embedding_size / n_head

        x = x.view(*new_shape)
        # Reorganiza el tensor sin cambiar datos (reshape lógico)

        return x.permute(0, 2, 1, 3)
        # Reordena dimensiones:
        # de (batch, seq, head, d_head)
        # a   (batch, head, seq, d_head)
        # Esto facilita calcular atención por cabeza


    def _attn(self, q, k, v, attn_mask=None):
        # Implementa la fórmula principal de scaled dot-product attention

        """Calcula atención usando Q, K, V"""

        scores = torch.matmul(q, k.transpose(-2, -1))
        # Q @ K^T
        # Calcula similitud entre tokens
        # Output shape: (batch, head, seq, seq)

        if self.scale:
            scores = scores / math.sqrt(v.size(-1))
            # Escalado por sqrt(dk)
            # Evita valores demasiado grandes en softmax
            # Mejora estabilidad numérica

        nd, ns = scores.size(-2), scores.size(-1)
        # nd = número de queries (seq length)
        # ns = número de keys (seq length)
        # En GPT usualmente nd = ns = T

        if attn_mask is not None:
            scores = scores + attn_mask
            # Aplica máscara de atención
            # Bloquea acceso a tokens futuros con valores muy negativos

        scores = self.softmax(scores)
        # Convierte scores en probabilidades normalizadas
        # Cada fila suma 1

        scores = self.dropout(scores)
        # Aplica dropout sobre attention weights
        # Reduce dependencia excesiva de ciertos tokens

        outputs = torch.matmul(scores, v)
        # Weighted sum:
        # combina valores V según importancia aprendida

        return outputs
        # Output final de atención por cabeza


    def merge_heads(self, x):
        # Une múltiples heads en un solo tensor

        x = x.permute(0, 2, 1, 3).contiguous()
        # Reordena de:
        # (batch, head, seq, d_head)
        # a (batch, seq, head, d_head)

        # contiguous() asegura memoria continua para reshape

        new_shape = x.size()[:-2] + (x.size(-2) * x.size(-1),)
        # Combina head y d_head en una sola dimensión:
        # head * d_head = d_model

        return x.view(*new_shape)
        # Devuelve tensor:
        # (batch, seq, d_model)


    def forward(self, x):
        # Forward pass completo de atención

        '''Calcula Q, K, V → split heads → attention → merge → projection'''

        x = self.c_attn(x)
        # Proyección lineal: genera Q, K, V concatenados
        # Shape: (batch, seq, 3 * d_model)

        q, k, v = x.split(self.d_model, dim=2)
        # Divide tensor en 3 partes iguales:
        # Query, Key, Value

        q, k, v = self.split_heads(q), self.split_heads(k), self.split_heads(v)
        # Divide cada uno en múltiples heads

        out = self._attn(q, k, v)
        # Calcula atención por cabeza

        out = self.merge_heads(out)
        # Une todas las heads en un solo tensor

        out = self.c_proj(out)
        # Proyección final para volver a d_model

        return out
        # Output final de la capa de atención

In [15]:
class TransformerBlock(nn.Module):
    # Define un bloque Transformer completo
    # Es la unidad básica repetida N veces en modelos tipo GPT

    def __init__(self, d_model=768, n_head=12, dropout=0.1):
        # Constructor del bloque Transformer

        super(TransformerBlock, self).__init__()
        # Inicializa la clase base nn.Module
        # Registra correctamente capas y parámetros en PyTorch

        self.attn = Attention(
            d_model=768,
            n_head=12,
            d_head=64,
            n_ctx=1024,
            bias=True,
            scale=False
        )
        # Capa de Self-Attention multi-head
        # Permite que cada token “mire” otros tokens de la secuencia
        # Aprende relaciones contextuales entre palabras

        self.feedforward = FeedForward(
            dropout=0.1,
            d_model=768,
            nx=768 * 4
        )
        # MLP (FeedForward network)
        # Procesa cada token de forma independiente después de atención
        # Expande dimensión (768 → 3072) y luego la reduce

        self.ln_1 = LayerNorm(d_model)
        # Layer Normalization antes de atención (pre-norm)
        # Normaliza activaciones para estabilizar entrenamiento

        self.ln_2 = LayerNorm(d_model)
        # Layer Normalization antes del feedforward
        # Reduce covariate shift y mejora convergencia


    def forward(self, x):
        # Define el flujo forward del bloque Transformer
        # x: tensor con shape (batch, sequence, d_model)

        # =====================================================
        # Attention sub-layer + Residual connection
        # =====================================================
        x = x + self.attn(self.ln_1(x))
        # ln_1(x):
        # normaliza entrada antes de atención (Pre-LN Transformer)

        # self.attn(...):
        # aplica self-attention multi-head

        # x + ...:
        # residual connection
        # ayuda a evitar vanishing gradients y preserva información original

        # =====================================================
        # FeedForward sub-layer + Residual connection
        # =====================================================
        x = x + self.feedforward(self.ln_2(x))
        # ln_2(x):
        # normaliza entrada antes del MLP

        # feedforward:
        # aplica transformación no lineal por token

        # x + ...:
        # segunda residual connection del bloque

        return x
        # Output final del bloque Transformer
        # Mantiene misma dimensión d_model para permitir stacking

In [16]:
class GPT2(nn.Module):
    # Define el modelo GPT-2 completo
    # Es un decoder-only Transformer para generación de texto autoregresiva

    def __init__(self, nlayers=12, n_ctx=1024, d_model=768, vcb_sz=50257):
        # Constructor del modelo GPT-2

        '''
        nlayers: número de bloques Transformer apilados (profundidad del modelo)
        n_ctx: contexto máximo (longitud máxima de secuencia de tokens)
        d_model: dimensión del embedding (tamaño del vector por token)
        vcb_sz: tamaño del vocabulario (cantidad de tokens posibles)
        '''

        super(GPT2, self).__init__()
        # Inicializa la clase base nn.Module
        # Registra parámetros y submódulos correctamente en PyTorch

        self.nlayers = nlayers
        # Guarda el número de capas Transformer

        block = TransformerBlock(
            d_model=768,
            n_head=12,
            dropout=0.1
        )
        # Define un bloque Transformer base
        # Este bloque será replicado N veces en la red

        self.h = _get_clones(block, 12)
        # Crea una lista de 12 copias independientes del TransformerBlock
        # Cada bloque tiene sus propios parámetros (no compartidos)

        self.wte = nn.Embedding(vcb_sz, d_model)
        # Word Token Embedding
        # Convierte IDs de tokens en vectores densos de tamaño d_model

        self.wpe = nn.Embedding(n_ctx, d_model)
        # Word Position Embedding
        # Añade información de posición en la secuencia (0..n_ctx-1)

        self.drop = nn.Dropout(0.1)
        # Dropout para regularización
        # Reduce overfitting apagando neuronas aleatoriamente

        self.ln_f = LayerNorm(d_model)
        # LayerNorm final
        # Estabiliza la salida antes de la proyección al vocabulario

        self.out = nn.Linear(d_model, vcb_sz, bias=False)
        # Capa final de proyección
        # Convierte embeddings → logits sobre vocabulario

        self.loss_fn = nn.CrossEntropyLoss()
        # Función de pérdida para clasificación multiclase
        # Se usa para predecir el siguiente token

        self.init_weights()
        # Inicializa todos los pesos del modelo


    def init_weights(self):
        # Inicialización de pesos del modelo

        self.out.weight = self.wte.weight
        # Weight tying:
        # comparte pesos entre embedding de entrada y capa de salida
        # mejora calidad del modelo y reduce parámetros

        self.apply(self._init_weights)
        # Aplica función de inicialización a todos los submódulos


    def _init_weights(self, module):
        # Regla de inicialización por tipo de módulo

        if isinstance(module, (nn.Linear, nn.Embedding, Conv1D)):
            # Si el módulo es una capa lineal, embedding o Conv1D

            module.weight.data.normal_(mean=0.0, std=0.02)
            # Inicializa pesos con distribución normal centrada en 0
            # std=0.02 mantiene valores pequeños para estabilidad

            if isinstance(module, (nn.Linear, Conv1D)) and module.bias is not None:
                # Si el módulo tiene bias (sesgo)

                module.bias.data.zero_()
                # Inicializa bias en cero
                # Evita sesgos iniciales en la red

        elif isinstance(module, nn.LayerNorm):
            # Si el módulo es LayerNorm

            module.bias.data.zero_()
            # Bias de LayerNorm inicializado en 0

            module.weight.data.fill_(1.0)
            # Weight de LayerNorm inicializado en 1
            # Mantiene identidad al inicio del entrenamiento


    def forward(self, src, labels=None, pos_ids=None):
        # Forward pass del modelo GPT-2

        '''
        src: input tokens (batch, sequence)
        labels: tokens objetivo para calcular loss (shifted targets)
        pos_ids: posiciones opcionales de tokens
        '''

        # =====================================================
        # POSITION IDS
        # =====================================================
        if pos_ids is None:
            pos_ids = torch.arange(0, src.size(-1)).unsqueeze(0)
            # Genera posiciones [0, 1, 2, ..., T-1]
            # unsqueeze(0) agrega dimensión batch

        pos_ids = pos_ids.to(src.device)
        # Mueve pos_ids al mismo device que el input (CPU o GPU)
        # Evita errores de mismatch de dispositivos

        # =====================================================
        # EMBEDDINGS
        # =====================================================
        inp = self.drop((self.wte(src) + self.wpe(pos_ids)))
        # wte(src): embedding de tokens
        # wpe(pos_ids): embedding de posiciones
        # suma ambos para combinar semántica + orden
        # dropout: regularización

        # =====================================================
        # TRANSFORMER STACK
        # =====================================================
        for i in range(self.nlayers):
            inp = self.h[i](inp)
            # pasa secuencialmente por cada TransformerBlock

        # =====================================================
        # FINAL LAYERS
        # =====================================================
        inp = self.ln_f(inp)
        # normalización final para estabilizar logits

        logits = self.out(inp)
        # proyección final a vocabulario
        # shape: (batch, seq, vocab_size)

        outputs = (logits,) + (inp,)
        # empaqueta outputs intermedios (logits + hidden states)

        # =====================================================
        # LOSS COMPUTATION (training mode)
        # =====================================================
        if labels is not None:
            # si se proporcionan labels → modo entrenamiento

            shift_logits = logits[..., :-1, :].contiguous()
            # elimina último token de predicción
            # porque no tiene target futuro

            shift_labels = labels[..., 1:].contiguous()
            # desplaza labels a la derecha
            # para predecir siguiente token

            loss = self.loss_fn(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )
            # cross entropy entre predicción y token real
            # aplana batch y secuencia

            outputs = (loss,) + outputs
            # añade loss al output

            return loss.mean()
            # retorna pérdida promedio

        # =====================================================
        # INFERENCE MODE
        # =====================================================
        return logits
        # devuelve logits para generación de texto

In [17]:
import torch.nn.functional as F
# Importa el módulo de funciones funcionales de PyTorch
# Incluye operaciones como activaciones, loss functions, softmax, etc.
# Se usa porque son funciones sin estado (no tienen parámetros entrenables)

import time
# Importa librería estándar de Python para manejo de tiempo
# Se usa normalmente para medir duración de entrenamiento o inferencia
# Ejemplo: time.time() para calcular elapsed time

from transformers import GPT2Tokenizer
# Importa el tokenizador GPT-2 desde la librería Hugging Face Transformers

# GPT2Tokenizer:
# Convierte texto (strings) en tokens numéricos que entiende el modelo
# También convierte tokens de vuelta a texto (decode)

# Este tokenizador usa Byte Pair Encoding (BPE)
# Lo que significa:
# - Divide palabras en subpalabras frecuentes
# - Maneja palabras nuevas sin romper el modelo

# Ejemplo conceptual:
# "playing" → ["play", "ing"] → [token_ids]

# Es un componente crítico porque:
# el modelo NO trabaja con texto directamente
# solo trabaja con IDs numéricos de tokens

In [18]:
model = GPT2()
# Crea una instancia del modelo GPT2 definido previamente
# Esto construye toda la arquitectura completa del Transformer

# GPT2():
# Ejecuta el constructor (__init__) de la clase GPT2
# Inicializa todos los componentes del modelo:

# - Embeddings de tokens (wte)
# - Embeddings de posición (wpe)
# - Stack de TransformerBlocks (self.h)
# - LayerNorm final (ln_f)
# - Capa de salida al vocabulario (out)
# - Función de pérdida (CrossEntropyLoss)
# - Inicialización de pesos (init_weights)

# Resultado:
# "model" ahora es un objeto entrenable de red neuronal profunda
# con parámetros inicializados y listo para entrenamiento o inferencia

# En este punto el modelo todavía NO ha aprendido nada
# solo tiene pesos inicializados aleatoriamente

In [19]:
!curl --output gpt2-pytorch_model.bin https://s3.amazonaws.com/models.huggingface.co/bert/gpt2-pytorch_model.bin
# Ejecuta un comando del sistema usando shell (por el símbolo ! en notebooks tipo Jupyter)

# curl:
# herramienta de línea de comandos para hacer requests HTTP/HTTPS
# se usa aquí para descargar un archivo desde internet

# --output gpt2-pytorch_model.bin:
# especifica el nombre del archivo donde se guardará la descarga
# en este caso: "gpt2-pytorch_model.bin"

# URL:
# https://s3.amazonaws.com/models.huggingface.co/bert/gpt2-pytorch_model.bin

# Es un modelo preentrenado de GPT-2 en formato binario (pesos del modelo)
# alojado en Amazon S3 (storage en la nube)

# ¿Qué contiene ese archivo?
# - weights (parámetros entrenados del modelo)
# - bias
# - matrices de atención
# - embeddings aprendidos

# IMPORTANTE:
# Este archivo NO es código
# es el estado entrenado de una red neuronal

# Uso típico después de descargarlo:
# model.load_state_dict(torch.load("gpt2-pytorch_model.bin"))

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  522M  100  522M    0     0  13.1M      0  0:00:39  0:00:39 --:--:-- 14.7M


In [20]:
model_dict = model.state_dict()
# state_dict():
# devuelve un diccionario con todos los parámetros del modelo
# incluye pesos (weights) y sesgos (biases) de cada capa

# model_dict:
# es la estructura de parámetros ACTUAL del modelo GPT2 creado por ti
# sirve como referencia de cómo deben llamarse las keys

state_dict = torch.load("./gpt2-pytorch_model.bin")
# torch.load():
# carga un archivo guardado en formato PyTorch (.bin o .pt)

# "./gpt2-pytorch_model.bin":
# archivo local que contiene pesos preentrenados de GPT-2

# state_dict:
# diccionario con los parámetros del modelo preentrenado
# contiene keys como:
# "transformer.h.0.attn.c_attn.weight", etc.


old_keys = []
new_keys = []
# listas vacías para almacenar:
# old_keys → nombres originales de parámetros en el modelo cargado
# new_keys → nombres corregidos compatibles con tu implementación


for key in state_dict.keys():
    # iteración sobre todas las keys del modelo preentrenado
    # cada key representa un tensor de pesos o bias

    if "mlp" in key:
        # verifica si el nombre del parámetro contiene "mlp"
        # en GPT-2 original de HuggingFace se usa nomenclatura "mlp"
        # pero en tu implementación se llama "feedforward"

        new_key = key.replace("mlp", "feedforward")
        # reemplaza el nombre "mlp" por "feedforward"
        # esto alinea los nombres del modelo preentrenado con tu arquitectura

        new_keys.append(new_key)
        # guarda el nuevo nombre corregido

        old_keys.append(key)
        # guarda el nombre original para referencia o debugging

# Resultado:
# estás creando un mapeo entre:
# modelo preentrenado (HuggingFace GPT-2)
# y tu implementación personalizada

In [21]:
for old_key, new_key in zip(old_keys, new_keys):
    # zip(old_keys, new_keys):
    # combina dos listas en pares (old_key, new_key)
    # cada par representa:
    # - nombre original del parámetro
    # - nombre corregido para tu modelo

    state_dict[new_key] = state_dict.pop(old_key)
    # state_dict.pop(old_key):
    # elimina la entrada con el nombre antiguo del diccionario
    # y devuelve su valor (tensor de pesos)

    # state_dict[new_key] = ...:
    # inserta ese mismo tensor bajo el nuevo nombre compatible con tu arquitectura

    # En resumen:
    # estás renombrando claves del state_dict
    # para que coincidan con tu implementación de GPT2

# Resultado final:
# state_dict ahora tiene nombres alineados con tu modelo
# listo para cargar con model.load_state_dict(state_dict)

In [22]:
pretrained_dict = {k: v for k, v in state_dict.items() if k in model_dict}
# Crea un nuevo diccionario filtrado usando dictionary comprehension
# k: clave del parámetro (nombre del tensor)
# v: valor del parámetro (tensor de pesos)

# state_dict.items():
# devuelve pares (key, value) de todos los parámetros del modelo preentrenado

# if k in model_dict:
# filtra solo los parámetros que existen en tu modelo actual
# evita errores por incompatibilidad de arquitectura

# Resultado:
# pretrained_dict contiene solo pesos compatibles con tu GPT2 personalizado


model_dict.update(pretrained_dict)
# update():
# actualiza el diccionario model_dict con los valores de pretrained_dict

# Si una key existe en ambos:
# el valor nuevo (preentrenado) reemplaza al antiguo (inicializado aleatoriamente)

# Resultado:
# model_dict ahora mezcla:
# - pesos preentrenados donde coinciden
# - pesos originales donde no hay coincidencia


model.load_state_dict(model_dict)
# Carga el diccionario de parámetros dentro del modelo PyTorch

# load_state_dict():
# asigna los pesos tensor por tensor a cada capa del modelo

# Si todo coincide correctamente:
# el modelo ahora tiene conocimiento aprendido previamente (GPT-2 pretrained)


model.eval()
# Cambia el modelo a modo evaluación (inference mode)

# Esto activa comportamientos específicos:
# - desactiva Dropout (no apaga neuronas aleatoriamente)
# - desactiva comportamiento de entrenamiento en BatchNorm (si existiera)

# Uso típico:
# cuando quieres generar texto o hacer inferencia, no entrenamiento

# Resultado final:
# el modelo queda listo para generar texto con pesos preentrenados

GPT2(
  (h): ModuleList(
    (0-11): 12 x TransformerBlock(
      (attn): Attention(
        (c_attn): Conv1D()
        (softmax): Softmax(dim=-1)
        (dropout): Dropout(p=0.1, inplace=False)
        (c_proj): Conv1D()
      )
      (feedforward): FeedForward(
        (c_fc): Conv1D()
        (c_proj): Conv1D()
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    )
  )
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (out): Linear(in_features=768, out_features=50257, bias=False)
  (loss_fn): CrossEntropyLoss()
)

In [23]:
total_params = sum(p.numel() for p in model.parameters())
# Calcula el número total de parámetros entrenables del modelo

# model.parameters():
# devuelve un iterable con todos los tensores de parámetros del modelo
# incluye weights y biases de todas las capas (Attention, FeedForward, Embeddings, etc.)

# p:
# cada p es un tensor individual de parámetros (por ejemplo una matriz de pesos)

# p.numel():
# “number of elements”
# devuelve cuántos valores numéricos contiene ese tensor
# ejemplo:
# tensor de shape (768, 768) → 768 * 768 elementos

# sum(...):
# suma todos los elementos de todos los tensores del modelo

# Resultado:
# total_params contiene un entero con el número total de parámetros del modelo
# útil para medir tamaño del modelo y complejidad

In [24]:
size_bytes = total_params * 4
# Calcula el tamaño total del modelo en bytes
# total_params: número total de parámetros del modelo
# 4: cada parámetro se asume como un float32 (32 bits = 4 bytes)

# Supuesto importante:
# PyTorch por defecto usa float32 para los pesos
# 1 float32 = 32 bits = 4 bytes


size_mb = size_bytes / (1024 ** 2)
# Convierte bytes a megabytes (MB)

# 1024 ** 2:
# 1 MB en computación = 1024 KB * 1024 bytes = 1,048,576 bytes

# Por eso:
# bytes / 1024^2 = MB


print(f"El tamaño total de GPT2 sin alteraciones es: {size_bytes} bytes o {size_mb:.2f} MB")
# Imprime el tamaño del modelo en dos formatos:

# {size_bytes}:
# tamaño exacto en bytes

# {size_mb:.2f}:
# tamaño en MB con 2 decimales

# f-string:
# permite interpolación directa de variables dentro del texto

# Resultado final:
# muestra cuánto espacio ocupa el modelo en memoria/disco

El tamaño total de GPT2 sin alteraciones es: 497759232 bytes o 474.70 MB


In [25]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
# Carga el tokenizador preentrenado GPT-2 desde Hugging Face
# from_pretrained("gpt2"):
# descarga y carga el vocabulario, reglas BPE y merges del modelo GPT-2

# GPT2Tokenizer:
# convierte texto (strings) en tokens numéricos (IDs)
# y también permite convertir IDs de vuelta a texto

# Este tokenizador usa Byte Pair Encoding (BPE):
# - divide palabras en sub-unidades frecuentes
# - maneja palabras desconocidas sin fallar
# - reduce tamaño del vocabulario manteniendo flexibilidad


context = torch.tensor([tokenizer.encode("The planet earth is a beautiful")])
# tokenizer.encode(...):
# convierte el texto en una lista de token IDs

# Ejemplo conceptual:
# "The planet earth is a beautiful" → [15496, 3959, 1850, 318, 257, 4950]

# torch.tensor([...]):
# convierte la lista de IDs en un tensor de PyTorch
# necesario porque el modelo GPT2 solo acepta tensores como entrada

# [ ... ] (doble lista):
# agrega dimensión batch (batch size = 1)

# Resultado:
# context es un tensor de forma:
# (1, sequence_length)
# listo para ser alimentado al modelo GPT2

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [26]:
def generate(context, ntok=550):
    # Define una función de generación autoregresiva de texto
    # context: secuencia inicial de tokens (prompt)
    # ntok: número de tokens a generar

    start_time = time.time()
    # Guarda el tiempo inicial para medir performance de inferencia

    for _ in range(ntok):
        # Bucle que genera token por token de forma secuencial (autoregresivo)

        out = model(context)
        # Pasa el contexto completo por el modelo GPT2
        # out tiene shape: (batch, seq_len, vocab_size)

        logits = out[:, -1, :]
        # Selecciona solo el último token de la secuencia
        # Es decir: predicción del siguiente token

        indices_to_remove = logits < torch.topk(logits, 10)[0][..., -1, None]
        # torch.topk(logits, 10):
        # obtiene los 10 logits más altos (top-k sampling)

        # [0]:
        # toma solo los valores (no los índices)

        # [..., -1, None]:
        # selecciona el décimo mejor valor como umbral

        # logits < threshold:
        # crea máscara booleana para todos los tokens por debajo del top-10

        # Resultado:
        # identifica tokens que NO están en el top-k

        logits[indices_to_remove] = -np.inf
        # reemplaza logits bajos con -infinito
        # esto hace que softmax los vuelva probabilidad ~0

        next_tok = torch.multinomial(
            F.softmax(logits, dim=-1),
            num_samples=1
        ).squeeze(1)
        # F.softmax(logits, dim=-1):
        # convierte logits en distribución de probabilidad

        # torch.multinomial:
        # muestrea un token según esa distribución (sampling estocástico)
        # permite diversidad en generación (no siempre el más probable)

        # num_samples=1:
        # selecciona un solo token

        # squeeze(1):
        # elimina dimensión extra del tensor

        context = torch.cat(
            [context, next_tok.unsqueeze(-1)],
            dim=-1
        )
        # concatena el nuevo token al contexto existente
        # next_tok.unsqueeze(-1):
        # convierte shape (batch,) → (batch, 1)

        # torch.cat:
        # extiende la secuencia con el nuevo token generado

    end_time = time.time()
    # Guarda tiempo final de ejecución

    inference_time = end_time - start_time
    # Calcula tiempo total de generación

    return context, inference_time
    # retorna:
    # - secuencia completa generada
    # - tiempo de inferencia

In [27]:
out, inference_time = generate(context, ntok=40)
# Llama la función generate()
# context: secuencia inicial de tokens (prompt ya tokenizado)
# ntok=40: indica que se generarán 40 tokens nuevos

# out:
# tensor con la secuencia completa (prompt + tokens generados)
# shape: (batch, sequence_total)

# inference_time:
# tiempo total de ejecución de la generación


decoded_output = tokenizer.decode(out[0])
# tokenizer.decode():
# convierte tokens numéricos (IDs) de vuelta a texto legible

# out[0]:
# selecciona el primer elemento del batch (batch size = 1)
# porque decode espera una sola secuencia

# decode:
# transforma:
# [token_ids] → string

# Ejemplo conceptual:
# [15496, 3959, 1850] → "The planet earth is ..."

# Resultado:
# decoded_output contiene el texto generado completo
# incluyendo prompt original + texto generado por el modelo

In [28]:
print(f"Inference Time: {inference_time:.4f} seconds")
# print():
# función de Python para mostrar información en consola

# f-string:
# permite insertar variables dentro del string usando {}

# inference_time:
# variable que contiene el tiempo total de generación del modelo

# :.4f:
# formatea el número float con 4 decimales
# ejemplo: 1.234567 → 1.2346

# Resultado:
# imprime cuánto tardó el modelo en generar texto


print(f"Generated Output: {decoded_output}")
# print():
# muestra el resultado final en consola

# decoded_output:
# texto generado por el modelo GPT-2 ya decodificado

# f-string:
# inserta directamente el string generado dentro del mensaje

# Resultado:
# muestra el prompt original + texto generado por el modelo

Inference Time: 6.9566 seconds
Generated Output: The planet earth is a beautiful place of beauty. of the world of the of the great, great place of the, of an of world. is the way of, of beauty, the world, of a the of is planet
